# Two coupled $0$-$\pi$: $n_{\theta 1}\otimes n_{\theta 2}$

In [1]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
from scipy.sparse.linalg import eigsh
import utils_2Q_gate_zp as ut
from joblib import Parallel, delayed
from IPython.display import display, Math
import pandas as pd
# ut.set_fig_font() ### Set various sizes in plotting

### Coupled zero pi : small-size basis for understanding

In [ ]:
truc1 = 10
zp = scq.Circuit(ut.zp_yml, from_file=False)
zp.configure(transformation_matrix=np.linalg.inv(ut.transform_2zeropi))
system_hierarchy = [[1,2],  [5,6]]
subsystem_trunc_dims = [10, 10]
zp.configure(system_hierarchy=system_hierarchy, subsystem_trunc_dims=subsystem_trunc_dims )

In [ ]:
zp.cutoff_ext_1, zp.cutoff_ext_5 = 5, 6
zp.cutoff_n_2, zp.cutoff_n_6 = 3, 4

eval0, eket0 = zp.subsystems[0].eigensys(evals_count=truc1)
eval1, eket1 = zp.subsystems[1].eigensys(evals_count=truc1)

sorted_idx0 = np.argsort(eval0)
eval0 = eval0[sorted_idx0]
eval0 = eval0 - eval0[0]
eket0 = ssp.csr_matrix([eket0[:,idx] for idx in range(truc1)])

sorted_idx1 = np.argsort(eval1)
eval1 = eval1[sorted_idx1]
eval1 = eval1 - eval1[0]
eket1 = ssp.csr_matrix([eket1[:,idx] for idx in range(truc1)])
n_theta0 = np.round(eket0 @ zp.subsystems[0].n2_operator() @ eket0.conj().T, 8).todense()
n_theta1 = np.round(eket1 @ zp.subsystems[1].n6_operator() @ eket1.conj().T, 8).todense()

In [3]:
zp.subsystems[0].n2_operator().todense()[:7,:7]

matrix([[-3.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0., -2.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0., -1.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  1.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  2.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  3.]])

In [4]:
zp.subsystems[0].n2_operator()

<35x35 sparse matrix of type '<class 'numpy.float64'>'
	with 30 stored elements in Compressed Sparse Column format>

### Coupled zero pi : large-size basis for convergence

In [8]:
zp.cutoff_ext_1, zp.cutoff_ext_5 = 100, 100
zp.cutoff_n_2, zp.cutoff_n_6 = 30, 30

eval0, eket0 = zp.subsystems[0].eigensys(evals_count=truc1)
eval1, eket1 = zp.subsystems[1].eigensys(evals_count=truc1)

sorted_idx0 = np.argsort(eval0)
eval0 = eval0[sorted_idx0]
eval0 = eval0 - eval0[0]
eket0 = ssp.csr_matrix([eket0[:,idx] for idx in range(truc1)])

sorted_idx1 = np.argsort(eval1)
eval1 = eval1[sorted_idx1]
eval1 = eval1 - eval1[0]
eket1 = ssp.csr_matrix([eket1[:,idx] for idx in range(truc1)])
n_theta0 = np.round(eket0 @ zp.subsystems[0].n2_operator() @ eket0.conj().T, 8).todense()
n_theta1 = np.round(eket1 @ zp.subsystems[1].n6_operator() @ eket1.conj().T, 8).todense()

In [18]:
qt.Qobj(n_theta0[[2,0]])

Quantum object: dims = [[2], [10]], shape = (2, 10), type = oper, isherm = False
Qobj data =
[[ 0.          0.01039018  0.          0.          0.          1.33426456
   0.          0.          0.06547216  0.        ]
 [ 0.         -1.35221235  0.          0.          0.          0.00440689
   0.          0.         -0.04376532  0.        ]]

In [ ]:
qt.Qobj(n_theta1[[2,0]])

Quantum object: dims = [[2], [10]], shape = (2, 10), type = oper, isherm = False
Qobj data =
[[ 0.          0.01882571  0.          0.          0.         -1.2855235
   0.          0.         -0.11680706  0.        ]
 [ 0.         -1.31210402  0.          0.          0.         -0.00762634
   0.          0.          0.04615249  0.        ]]

### Single zero pi

In [ ]:
def zero_pi_initialize(EJ=6.013, truncation=10, ncut=30, phi_cut=100):

    # Define system parameters (in GHz)
    EL = 0.377  # Inductive energy
    # EJ = 6.013  # Josephson energy
    EC_phi = 1.142  # Phi mode charging energy
    EC_theta = 0.092  # Theta mode charging energy

    # Compute derived parameters
    E_CJ = 2 * EC_phi
    E_C = 2 / (1 / EC_theta - 1 / EC_phi)

    # Create the grid for the phi coordinate
    phi_grid = scq.Grid1d(-6 * np.pi, 6 * np.pi, phi_cut)

    # Initialize the Zero-Pi qubit system
    zero_pi = scq.ZeroPi(
        grid=phi_grid,
        EJ=EJ,
        EL=EL,
        ECJ=E_CJ,
        EC=E_C,
        dEJ=0.0,
        ng=0.0,
        flux=0.0,
        ncut=ncut,
        truncated_dim=truncation,
    )

    # Compute matrix elements for the theta and phi operators
    n_Theta = zero_pi.matrixelement_table(operator="n_theta_operator", evals_count=truncation)
    return n_Theta


In [17]:
n_Theta = zero_pi_initialize(EJ=6.013)
qt.Qobj(n_Theta[[2,0]]).tidyup()

Quantum object: dims = [[2], [10]], shape = (2, 10), type = oper, isherm = False
Qobj data =
[[ 0.         -0.03654567  0.          0.          0.          1.2618861
   0.          0.19081532  0.          0.        ]
 [ 0.          1.30442915  0.          0.          0.          0.01492428
   0.         -0.05224724  0.          0.        ]]

In [20]:
n_Theta = zero_pi_initialize(EJ=6.013*0.9)
qt.Qobj(n_Theta[[2,0]]).tidyup()

Quantum object: dims = [[2], [10]], shape = (2, 10), type = oper, isherm = False
Qobj data =
[[ 0.         -0.06267093  0.          0.          0.         -1.18457846
   0.          0.30312282  0.          0.        ]
 [ 0.          1.2631747   0.          0.          0.         -0.02379564
   0.         -0.05387746  0.          0.        ]]

### Full code for coupled zero pi

In [ ]:
def get_operator_two_zeropi(Ec0=1.0, truc1=30, truc_tot=50, charge_pick=False, n_cut=60, phi_cut=200):
    zp = scq.Circuit(zp_yml, from_file=False)
    zp.Ec0 = Ec0
    zp.configure(transformation_matrix=np.linalg.inv(transform_2zeropi))

    ##############################################################################################
    ### Construct subsystem, calculate eigenvalues
    system_hierarchy = [[1,2],  [5,6]]
    subsystem_trunc_dims = [100, 100]
    zp.configure(system_hierarchy=system_hierarchy,
                subsystem_trunc_dims=subsystem_trunc_dims)

    zp.cutoff_ext_1, zp.cutoff_ext_5 = phi_cut, phi_cut
    zp.cutoff_n_2, zp.cutoff_n_6 = n_cut, n_cut

    ### the two-line code below takes time when truc1 is large
    eval0, eket0 = zp.subsystems[0].eigensys(evals_count=truc1)
    eval1, eket1 = zp.subsystems[1].eigensys(evals_count=truc1)

    sorted_idx0 = np.argsort(eval0)
    eval0 = eval0[sorted_idx0]
    eval0 = eval0 - eval0[0]
    eket0 = ssp.csr_matrix([eket0[:,idx] for idx in range(truc1)])

    sorted_idx1 = np.argsort(eval1)
    eval1 = eval1[sorted_idx1]
    eval1 = eval1 - eval1[0]
    eket1 = ssp.csr_matrix([eket1[:,idx] for idx in range(truc1)])

    # get the n-operator in qubit basis of single qubit
    n_theta0 = (eket0 @ zp.subsystems[0].n2_operator() @ eket0.conj().T).todense()
    n_theta1 = (eket1 @ zp.subsystems[1].n6_operator() @ eket1.conj().T).todense()
    ##############################################################################################
    ###  Truncate two qubits using charge matrix elements
    hspace_0 = np.arange(truc1)
    hspace_1 = np.arange(truc1)
    thresh_matrix_element=1e-4
    if charge_pick:
        hspace_0 = [0, 2]
        hspace_1 = [0, 2]
        for s in hspace_0:
            for i in range(truc1):
                if np.abs(n_theta0[s, i]) > thresh_matrix_element and i not in hspace_0:
                    hspace_0.append(i)
        hspace_0.sort()
        for s in hspace_1:
            for i in range(truc1):
                if np.abs(n_theta1[s, i]) > thresh_matrix_element and i not in hspace_1:
                    hspace_1.append(i)
        hspace_1.sort()
        # if hspace_theta != None:
        #     hspace_0 = hspace_theta
        #     hspace_1 = hspace_theta
        n_theta0 = truncate_2(n_theta0, hspace_0)
        n_theta1 = truncate_2(n_theta1, hspace_1)
        eval0 = eval0[hspace_0]
        eval1 = eval1[hspace_1]
        # eket0 = eket0[hspace_0]
        # eket1 = eket1[hspace_1]

    ##############################################################################################
    ###  Compute eigenvalues and eigenvectors for coupling H
    n2, n6 = symbols('n2 n6')
    g = float(zp.sym_interaction((1,0), return_expr=True).coeff(n2*n6) )
    Hint = qt.tensor(qt.Qobj(n_theta0) , qt.Qobj(n_theta1))
    H_bare = (  qt.tensor(qt.Qobj(np.diag(eval0)),  qt.identity(len(hspace_1)))
            +  qt.tensor(qt.identity(len(hspace_0)),  qt.Qobj(np.diag(eval1))) )
    # Htot = (g* Hint + H_bare).tidyup(atol=1e-8)
    Htot = g* Hint + H_bare

    ### the one-line code below takes time when truc1 is large
    k = Htot.shape[0] - 1
    if truc_tot != None:
        k = truc_tot
    eval_tot, eket_tot = ssp.linalg.eigsh(Htot.data, k=k, which='SA', tol=1.e-10)

    sorted_idx_tot = np.argsort(eval_tot)
    eval_tot = eval_tot[sorted_idx_tot]
    eval_tot = eval_tot - eval_tot[0]
    eket_tot = ssp.csr_matrix([eket_tot[:,idx] for idx in sorted_idx_tot])

    ##############################################################################################
    ###  Get wavefunction overlap for the truncated dressed states
    bare_state = [[qt.tensor(qt.basis(len(hspace_0), i), qt.basis(len(hspace_1), j))
                            for j in range(len(hspace_1))]
                                for i in range(len(hspace_0))]
    def find_overlap(eket):
        overlaps = np.array([[np.abs( (eket @ bare_state[i][j].data).todense()[0,0] )
                            for j in range(len(eval1))]
                                for i in range(len(eval0))])
        flat_array = overlaps.flatten() # Flatten the 2D array
        # Find the indices of the top 3 largest values (in the flattened 1D array)
        top_indices_flat = np.argpartition(-flat_array, 10)[:10]
        # Convert the flat indices to 2D indices
        top_indices_2d = np.unravel_index(top_indices_flat, overlaps.shape)
        # Extract the values corresponding to the indices
        top_values = overlaps[top_indices_2d]
        # Sort the values in descending order
        sorted_indices = np.argsort(-top_values)  # Use a negative sign for descending order
        sorted_top_indices = [tuple(zip(top_indices_2d[0], top_indices_2d[1]))[i] for i in sorted_indices]
        sorted_top_values = top_values[sorted_indices]
        return sorted_top_indices, sorted_top_values

    result = Parallel(n_jobs=10, verbose=0)(delayed(find_overlap)(arg) for arg in eket_tot)
    top_index = [result[i][0] for i in range(eval_tot.shape[0])]
    top_overlap = [result[i][1] for i in range(eval_tot.shape[0])]
    # for i in range(truc_tot):
    #     print(i, top3_index[i], top3_overlap[i])

    ##############################################################################################
    ### Get the dressed states index
    index_array = [] # array index in each qubit (# in hspace_0, hspace_1)
    for i, index in enumerate(top_index):
        j=0
        while j < len(index):
            if index[j] not in index_array:
                index_array.append(index[j])
                break
            else:
                j+=1
            if j==10:
                index_array.append((0,0))
                print(i, 'need to further compare overlap')
    hspace_full = [(str(hspace_0[idx[0]])+'-'+str(hspace_1[idx[1]])) for idx in index_array] # actual state index in each qubit

    n_theta0_dress = ssp.kron(n_theta0, ssp.identity(len(hspace_1)))
    n_theta0_dress = np.abs(np.round(eket_tot @ n_theta0_dress @ eket_tot.conj().T, 8)).todense()
    n_theta1_dress = ssp.kron(ssp.identity(len(hspace_0)), n_theta1)
    n_theta1_dress = np.abs(np.round(eket_tot @ n_theta1_dress @ eket_tot.conj().T, 8)).todense()